# 4 - Merge

Join the CHR life-expectancy outcomes with the Census ACS demographics on 5-digit FIPS code.
Includes `pct_native_american` and population density from the Census Gazetteer.

In [1]:
import pandas as pd

outcomes = pd.read_csv('data/county_outcomes.csv', dtype={'fips': str})
census   = pd.read_csv('data/census_df.csv',       dtype={'fips': str})

outcomes['fips'] = outcomes['fips'].str.zfill(5)
census['fips']   = census['fips'].str.zfill(5)

print('outcomes:', outcomes.shape, ' census:', census.shape)
print('census columns:', list(census.columns))

outcomes: (3070, 4)  census: (3144, 10)
census columns: ['fips', 'name', 'median_hh_income', 'median_age', 'population', 'log_population', 'pct_bachelors_plus', 'pct_poverty', 'pct_nhwhite', 'pct_native_american']


In [2]:
merged = outcomes.merge(census, on='fips', how='inner')
print('merged inner-join:', merged.shape)
merged.head()

merged inner-join: (3062, 13)


,fips,county_name,state_name,life_expectancy,name,median_hh_income,median_age,population,log_population,pct_bachelors_plus,pct_poverty,pct_nhwhite,pct_native_american
0,01001,Autauga County,Alabama,75.263497,"Autauga County, Alabama",68315,39.0,58761,4.769089,29.558575,11.373969,72.556628,0.100407
1,01003,Baldwin County,Alabama,76.738314,"Baldwin County, Alabama",71039,43.7,233420,5.368138,32.561579,10.213951,82.324137,0.219347
2,01005,Barbour County,Alabama,72.377024,"Barbour County, Alabama",39712,40.6,24877,4.395798,11.881188,24.163654,44.555212,0.253246
3,01007,Bibb County,Alabama,72.251369,"Bibb County, Alabama",50669,40.3,22251,4.347350,10.919937,20.622960,74.243854,0.067413
4,01009,Blount County,Alabama,73.376568,"Blount County, Alabama",57440,40.8,59077,4.771418,14.741407,14.173188,85.674628,0.126953


In [3]:
# Join land area and population density from Census Gazetteer
density = pd.read_csv('data/pop_density.csv', dtype={'fips': str})
density['fips'] = density['fips'].str.zfill(5)
density = density[['fips', 'land_sqmi', 'pop_density']]

merged = merged.merge(density, on='fips', how='left')
print('after density join:', merged.shape)
print(f"  land_sqmi nulls: {merged['land_sqmi'].isna().sum()}")
merged[['fips', 'county_name', 'land_sqmi', 'pop_density']].head()

after density join: (3062, 15)
  land_sqmi nulls: 0


,fips,county_name,land_sqmi,pop_density
0,01001,Autauga County,594.455,104.162636
1,01003,Baldwin County,1589.943,168.409182
2,01005,Barbour County,885.008,27.804268
3,01007,Bibb County,622.470,35.763973
4,01009,Blount County,644.905,93.278855


### Diagnose merge losses

In [4]:
only_outcomes = set(outcomes['fips']) - set(census['fips'])
only_census   = set(census['fips']) - set(outcomes['fips'])
print(f'In outcomes but not census: {len(only_outcomes)}')
print(f'In census but not outcomes: {len(only_census)}')
if only_outcomes:
    print('  Examples:', list(only_outcomes)[:5])
if only_census:
    print('  Examples:', list(only_census)[:5])

In outcomes but not census: 8
In census but not outcomes: 82
  Examples: ['09015', '09011', '09001', '09013', '09009']
  Examples: ['48431', '31183', '46021', '02063', '31113']


In [5]:
merged.to_csv('data/merged_df.csv', index=False)
print('Wrote data/merged_df.csv -', len(merged), 'counties')

Wrote data/merged_df.csv - 3062 counties
